# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrinidhi-02/flyrank-machine-learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Unit of analysis:** One row represents one pseudonymized content item (page). The starter dataset contains 30,000 rows and covers 32 pseudonymized clients.

**Time window:** The activity metrics are aggregated over a trailing 90-day window ending at the dataset export time. `content_age_days` represents the number of days since the content was created. The dataset does not establish an exact calendar start and end date for every row from this information alone.

In [8]:

import pandas as pd

url = "https://raw.githubusercontent.com/Shrinidhi-02/flyrank-machine-learning/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

print("\nContent age in days:")
print(df["content_age_days"].describe())

print("\nDataset columns:")
print(df.columns.tolist())


Total rows: 30000
Total columns: 44
Unique content items: 30000
Unique clients: 32
Duplicate content IDs: 0

Content age in days:
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64

Dataset columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_posit

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Features:** Candidate input fields include `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, and `char_count`. These describe the content or its search context and can be considered as potential predictors.

**Label:** The target is `is_declining_label`, where 1 means the content is classified as declining and 0 means it is not. This label is derived from `trend_direction`.

**Context:** `content_id` and `client_id` identify content items and clients. They are useful for grouping, checking data, and splitting data, but should not be model features.

**Excluded fields:** `trend_direction` and `trend_pct` must be excluded from model features because they are directly related to how the target label is created. Using them would cause data leakage. Also, `avg_position = 0` represents missing position data, not an actual rank of zero.

**Missing values:** Missing data should be investigated before deciding how to handle it. Missingness may depend on `content_type`, so missing values should not automatically be replaced with zero.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Check dataset size and unique identifiers
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

# 2. Check whether the target label exists
print("\nTarget label check:")
print("is_declining_label exists:", "is_declining_label" in df.columns)

# 3. Check missing values in every column
print("\nMissing values by column:")
print(df.isna().sum().sort_values(ascending=False).to_string())

# 4. Check missing values by content type
if "content_type" in df.columns and "avg_position" in df.columns:
    print("\nMissing avg_position by content type:")
    print(df.groupby("content_type")["avg_position"].apply(
        lambda s: s.isna().sum()
    ).to_string())

# 5. Check content age range
print("\nContent age range:")
print("Minimum:", df["content_age_days"].min())
print("Maximum:", df["content_age_days"].max())

# 6. Check special avg_position value
if "avg_position" in df.columns:
    print("\nRows where avg_position is zero:",
          (df["avg_position"] == 0).sum())

Rows: 30000
Columns: 44
Unique content IDs: 30000
Unique clients: 32
Duplicate content IDs: 0

Target label check:
is_declining_label exists: False

Missing values by column:
provider_used             21438
word_count                 7699
char_count                 7699
word_count_tier            7699
char_count_tier            7699
model_used                 5733
trend_pct                  3388
competition_level          2610
search_volume              2468
cpc                        2468
competition                2468
main_intent                2374
scroll_rate                 125
content_type                  0
client_id                     0
content_id                    0
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
users_90d                     0
engaged_sessions_90d     

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Data limitations:** This dataset is a snapshot with one row per content item and aggregated metrics over a trailing 90-day window. It does not provide a complete daily history for every content item, so I cannot verify balanced history or reconstruct every daily change from this CSV alone.

The starter dataset contains 30,000 content items across 32 pseudonymized clients. Results may not generalize to all clients or websites. Missing values are present in fields such as `provider_used`, `word_count`, and `search_volume`, and missingness may follow patterns rather than occur randomly.

The `trend_direction` field is available, but `is_declining_label` is not present in this raw CSV. The label must be created or obtained separately before it can be used as a prediction target. `trend_pct` and `trend_direction` must not be used as input features because they are related to the label definition.

The 90-day aggregate metrics and the last-30-day and previous-30-day metrics cover overlapping periods. Therefore, they are not independent observations. This dataset can support exploratory analysis and directional decision support, but it cannot establish that a particular feature caused content to decline.

In [12]:
print("DATA LIMITS CHECK")

print("\nDataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())

print("\nTarget available in raw dataset:",
      "is_declining_label" in df.columns)

print("\nTrend direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nZero avg_position rows:",
      (df["avg_position"] == 0).sum())

print("\nRows with missing trend_pct:",
      df["trend_pct"].isna().sum())

print("\nContent age range:",
      df["content_age_days"].min(),
      "to",
      df["content_age_days"].max(),
      "days")

DATA LIMITS CHECK

Dataset shape: (30000, 44)
Number of clients: 32

Target available in raw dataset: False

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Zero avg_position rows: 1205

Rows with missing trend_pct: 3388

Content age range: 90 to 564 days


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.